### Step 1: Automated collection of product pages

In [5]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime

In [8]:
# Website URL

BASE_URL = "https://scrapingsandbox.com/"

response = requests.get(BASE_URL, timeout=30)

print("Status Code:", response.status_code)
print("Website Accessible:", response.status_code == 200)

Status Code: 200
Website Accessible: True


In [9]:
response = session.get(BASE_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

In [10]:
product_links = soup.find_all(
    "a",
    href=lambda href: href and href.startswith("/product/")
)

product_urls = sorted({
    urljoin(BASE_URL, link["href"])
    for link in product_links
})

print("Product URLs found on first page:", len(product_urls))


Product URLs found on first page: 24


In [11]:
product_urls = [
    urljoin(BASE_URL, f"product/{product_id}")
    for product_id in range(1, 501)
]

In [12]:
raw_products = []

for index, product_url in enumerate(product_urls, start=1):
    try:
        page = session.get(product_url, timeout=30)
        page.raise_for_status()

        product_soup = BeautifulSoup(page.text, "html.parser")

        raw_text = product_soup.get_text(
            separator=" ",
            strip=True
        )

        raw_products.append({
            "Product_URL": product_url,
            "Raw_Text": raw_text,
            "Scraped_Date": datetime.now().isoformat()
        })

        if index % 25 == 0:
            print(f"Collected: {index}/500")

        time.sleep(0.2)

    except requests.RequestException as error:
        print(f"Failed: {product_url} | {error}")

Collected: 25/500
Collected: 50/500
Collected: 75/500
Collected: 100/500
Collected: 125/500
Collected: 150/500
Collected: 175/500
Collected: 200/500
Collected: 225/500
Collected: 250/500
Collected: 275/500
Collected: 300/500
Collected: 325/500
Collected: 350/500
Collected: 375/500
Collected: 400/500
Collected: 425/500
Collected: 450/500
Collected: 475/500
Collected: 500/500


In [13]:
raw_df = pd.DataFrame(raw_products)


In [14]:
raw_df.to_csv(
    "raw_ecommerce_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nScraping completed!")
print("Collected pages:", len(raw_df))
print("Dataset shape:", raw_df.shape)
print("\nFirst product raw text:")
if not raw_df.empty:
    print(raw_df.loc[0, "Raw_Text"])



Scraping completed!
Collected pages: 500
Dataset shape: (500, 3)

First product raw text:
Lightweight Probiotics - Scraping Sandbox Scraping Sandbox v1.0 Playground by Agenty Back to products SoundWave Health Lightweight Probiotics 4 ( 52 reviews) ● In Stock $ 155.62 $ 206.69 25 % OFF High-quality lightweight probiotics from SoundWave. Perfect for everyday use. Features premium materials and thoughtful design. This health product has been carefully crafted to meet the highest standards of quality and performance. SKU SKU-HEA-0001 Scraped 8/29/2022 Tags natural fitness Scraped JSON Data Copy {
  "id": 1,
  "title": "Lightweight Probiotics",
  "vendor": "SoundWave",
  "category": "Health",
  "price": 155.62,
  "compareAtPrice": 206.69,
  "description": "High-quality lightweight probiotics from SoundWave. Perfect for everyday use. Features premium materials and thoughtful design. This health product has been carefully crafted to meet the highest standards of quality and performance.",
  

### Step 2: Extract structured product data

In [16]:
import json

product_records = []
variant_records = []
extraction_errors = []

decoder = json.JSONDecoder()

for index, row in raw_df.iterrows():

    raw_text = row["Raw_Text"]

    try:
        # Locate the embedded JSON data
        marker = "Scraped JSON Data Copy"
        json_position = raw_text.find(marker)

        if json_position == -1:
            raise ValueError("JSON marker not found")

        json_text = raw_text[
            json_position + len(marker):
        ].strip()

        # Decode JSON object
        json_start = json_text.find("{")

        if json_start == -1:
            raise ValueError("JSON object not found")

        product_data, end_position = decoder.raw_decode(
            json_text,
            json_start
        )

        # Extract main product fields
        product_records.append({
            "Product_ID": product_data.get("id"),
            "Product_Name": product_data.get("title"),
            "Vendor": product_data.get("vendor"),
            "Category": product_data.get("category"),
            "Price": product_data.get("price"),
            "Original_Price": product_data.get("compareAtPrice"),
            "Description": product_data.get("description"),
            "SKU": product_data.get("sku"),
            "Availability": product_data.get("inStock"),
            "Rating": product_data.get("rating"),
            "Reviews_Count": product_data.get("reviewCount"),
            "Tags": ", ".join(product_data.get("tags", [])),
            "Created_Date": product_data.get("createdAt"),
            "Product_URL": row["Product_URL"],
            "Scraped_Date": row["Scraped_Date"]
        })

        # Extract product variants separately
        for variant in product_data.get("variants", []):
            variant_records.append({
                "Product_ID": product_data.get("id"),
                "Product_Name": product_data.get("title"),
                "Variant_Color": variant.get("color"),
                "Variant_Size": variant.get("size"),
                "Variant_SKU": variant.get("sku"),
                "Variant_Price": variant.get("price"),
                "Variant_Availability": variant.get("inStock")
            })

    except (ValueError, json.JSONDecodeError, TypeError) as error:
        extraction_errors.append({
            "Product_URL": row["Product_URL"],
            "Error": str(error)
        })

# Create structured DataFrames
products_df = pd.DataFrame(product_records)
variants_df = pd.DataFrame(variant_records)

print("Products extracted:", len(products_df))
print("Variants extracted:", len(variants_df))
print("Extraction errors:", len(extraction_errors))

print("\nProducts Dataset Shape:", products_df.shape)
print("Variants Dataset Shape:", variants_df.shape)

print("\nProducts Preview:")
display(products_df.head())

print("\nVariants Preview:")
display(variants_df.head())


Products extracted: 500
Variants extracted: 1685
Extraction errors: 0

Products Dataset Shape: (500, 15)
Variants Dataset Shape: (1685, 7)

Products Preview:


,Product_ID,Product_Name,Vendor,Category,Price,Original_Price,Description,SKU,Availability,Rating,Reviews_Count,Tags,Created_Date,Product_URL,Scraped_Date
0,1,Lightweight Probiotics,SoundWave,Health,155.62,206.69,High-quality lightweight probiotics from Sound...,SKU-HEA-0001,True,4.0,52,"natural, fitness",2022-08-29T00:00:00.000Z,https://scrapingsandbox.com/product/1,2026-11-06T15:30:12.447010
1,2,Wireless LED Desk Lamp,SoundWave,Electronics,145.40,210.01,High-quality wireless led desk lamp from Sound...,SKU-ELE-0002,False,4.6,12,"tech, digital, USB",2023-06-04T00:00:00.000Z,https://scrapingsandbox.com/product/2,2026-11-06T15:30:12.792686
2,3,Deluxe Tea Collection,StyleCraft,Food & Beverage,185.01,313.98,High-quality deluxe tea collection from StyleC...,SKU-FOO-0003,True,3.5,211,organic,2023-04-17T00:00:00.000Z,https://scrapingsandbox.com/product/3,2026-11-06T15:30:13.145211
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,High-quality essential tire gauge from TechVau...,SKU-AUT-0004,False,3.6,47,"car, accessories, travel",2022-10-24T00:00:00.000Z,https://scrapingsandbox.com/product/4,2026-11-06T15:30:13.497748
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,High-quality smart carving tools from SparkleB...,SKU-ART-0005,True,3.5,353,"drawing, painting",2022-08-07T00:00:00.000Z,https://scrapingsandbox.com/product/5,2026-11-06T15:30:13.852708



Variants Preview:


,Product_ID,Product_Name,Variant_Color,Variant_Size,Variant_SKU,Variant_Price,Variant_Availability
0,1,Lightweight Probiotics,Pink,XXL,SKU-HEA-0001-PIN-XXL,153.43,True
1,1,Lightweight Probiotics,Pink,XS,SKU-HEA-0001-PIN-XS,160.25,True
2,1,Lightweight Probiotics,White,L,SKU-HEA-0001-WHI-L,153.31,False
3,1,Lightweight Probiotics,Navy,XL,SKU-HEA-0001-NAV-XL,155.79,True
4,2,Wireless LED Desk Lamp,Orange,S,SKU-ELE-0002-ORA-S,146.54,True


### Step 3: Save the extracted data

In [17]:

products_df.to_csv(
    "ecommerce_products.csv",
    index=False,
    encoding="utf-8-sig"
)

variants_df.to_csv(
    "ecommerce_variants.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Products and variants exported successfully!")


Products and variants exported successfully!


### Step 4: Data Cleaning & Quality Validation

#### 1. Dataset Structure & Data Types

In [18]:
print("PRODUCTS DATASET")
print("=" * 50)

products_df.info()

print("\nVARIANTS DATASET")
print("=" * 50)

variants_df.info()

PRODUCTS DATASET
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product_ID      500 non-null    int64  
 1   Product_Name    500 non-null    str    
 2   Vendor          500 non-null    str    
 3   Category        500 non-null    str    
 4   Price           500 non-null    float64
 5   Original_Price  189 non-null    float64
 6   Description     500 non-null    str    
 7   SKU             500 non-null    str    
 8   Availability    500 non-null    bool   
 9   Rating          500 non-null    float64
 10  Reviews_Count   500 non-null    int64  
 11  Tags            500 non-null    str    
 12  Created_Date    500 non-null    str    
 13  Product_URL     500 non-null    str    
 14  Scraped_Date    500 non-null    str    
dtypes: bool(1), float64(3), int64(2), str(9)
memory usage: 245.3 KB

VARIANTS DATASET
<class 'pandas.DataFrame'>
RangeIndex

#### 2. Missing Values Analysis

In [19]:
print("Products Missing Values:")
print(products_df.isnull().sum())

print("\nVariants Missing Values:")
print(variants_df.isnull().sum())

print("\nMissing Values Percentage:")
missing_percentage = (
    products_df.isnull().mean() * 100
).round(2)

print(missing_percentage)

Products Missing Values:
Product_ID          0
Product_Name        0
Vendor              0
Category            0
Price               0
Original_Price    311
Description         0
SKU                 0
Availability        0
Rating              0
Reviews_Count       0
Tags                0
Created_Date        0
Product_URL         0
Scraped_Date        0
dtype: int64

Variants Missing Values:
Product_ID              0
Product_Name            0
Variant_Color           0
Variant_Size            0
Variant_SKU             0
Variant_Price           0
Variant_Availability    0
dtype: int64

Missing Values Percentage:
Product_ID         0.0
Product_Name       0.0
Vendor             0.0
Category           0.0
Price              0.0
Original_Price    62.2
Description        0.0
SKU                0.0
Availability       0.0
Rating             0.0
Reviews_Count      0.0
Tags               0.0
Created_Date       0.0
Product_URL        0.0
Scraped_Date       0.0
dtype: float64


#### 3. Duplicate Records

In [20]:
print("Duplicate Product Rows:",
      products_df.duplicated().sum())

print("Duplicate Product IDs:",
      products_df["Product_ID"].duplicated().sum())

print("Duplicate Product SKUs:",
      products_df["SKU"].duplicated().sum())

print("\nDuplicate Variant Rows:",
      variants_df.duplicated().sum())

print("Duplicate Variant SKUs:",
      variants_df["Variant_SKU"].duplicated().sum())

Duplicate Product Rows: 0
Duplicate Product IDs: 0
Duplicate Product SKUs: 0

Duplicate Variant Rows: 0
Duplicate Variant SKUs: 0


#### 4. Business Rules Validation

In [21]:
print("Invalid Prices:")
print(
    products_df[
        (products_df["Price"].isna()) |
        (products_df["Price"] <= 0)
    ][["Product_ID", "Product_Name", "Price"]]
)

print("\nInvalid Ratings:")
print(
    products_df[
        (products_df["Rating"] < 0) |
        (products_df["Rating"] > 5)
    ][["Product_ID", "Rating"]]
)

print("\nInvalid Reviews Count:")
print(
    products_df[
        products_df["Reviews_Count"] < 0
    ][["Product_ID", "Reviews_Count"]]
)

print("\nOriginal Price Lower Than Current Price:")
print(
    products_df[
        products_df["Original_Price"].notna() &
        (products_df["Original_Price"] < products_df["Price"])
    ][["Product_Name", "Price", "Original_Price"]]
)

Invalid Prices:
Empty DataFrame
Columns: [Product_ID, Product_Name, Price]
Index: []

Invalid Ratings:
Empty DataFrame
Columns: [Product_ID, Rating]
Index: []

Invalid Reviews Count:
Empty DataFrame
Columns: [Product_ID, Reviews_Count]
Index: []

Original Price Lower Than Current Price:
Empty DataFrame
Columns: [Product_Name, Price, Original_Price]
Index: []


#### 5. Check Variants Relationship

In [22]:
product_ids = set(products_df["Product_ID"])

orphan_variants = variants_df[
    ~variants_df["Product_ID"].isin(product_ids)
]

print("Variants without matching products:",
      len(orphan_variants))

print("\nVariants per Product:")
print(
    variants_df.groupby("Product_ID")
    .size()
    .describe()
)

Variants without matching products: 0

Variants per Product:
count    500.000000
mean       3.370000
std        1.097045
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000
dtype: float64


#### 6. Generate Quality Summary

In [23]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Total Products",
        "Total Variants",
        "Missing Product Values",
        "Duplicate Product Rows",
        "Duplicate Product IDs",
        "Duplicate Product SKUs",
        "Duplicate Variant SKUs",
        "Orphan Variants"
    ],
    "Value": [
        len(products_df),
        len(variants_df),
        products_df.isnull().sum().sum(),
        products_df.duplicated().sum(),
        products_df["Product_ID"].duplicated().sum(),
        products_df["SKU"].duplicated().sum(),
        variants_df["Variant_SKU"].duplicated().sum(),
        len(orphan_variants)
    ]
})

display(quality_summary)

,Metric,Value
0,Total Products,500
1,Total Variants,1685
2,Missing Product Values,311
3,Duplicate Product Rows,0
4,Duplicate Product IDs,0
5,Duplicate Product SKUs,0
6,Duplicate Variant SKUs,0
7,Orphan Variants,0


### Step 5: Identify Missing Values

In [24]:
missing_report = pd.DataFrame({
    "Missing_Count": products_df.isnull().sum(),
    "Missing_Percentage": (
        products_df.isnull().mean() * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    by="Missing_Count",
    ascending=False
)

display(missing_report)


,Missing_Count,Missing_Percentage
Original_Price,311,62.2
Product_Name,0,0.0
Product_ID,0,0.0
Vendor,0,0.0
Category,0,0.0
Price,0,0.0
Description,0,0.0
SKU,0,0.0
Availability,0,0.0
Rating,0,0.0


### Step 6: Inspect Missing Records

In [25]:
missing_products = products_df[
    products_df.isnull().any(axis=1)
]

print("Products with missing values:",
      len(missing_products))

display(missing_products.head(15))


Products with missing values: 311


,Product_ID,Product_Name,Vendor,Category,Price,Original_Price,Description,SKU,Availability,Rating,Reviews_Count,Tags,Created_Date,Product_URL,Scraped_Date
3,4,Essential Tire Gauge,TechVault,Automotive,70.86,NaN,High-quality essential tire gauge from TechVau...,SKU-AUT-0004,False,3.6,47,"car, accessories, travel",2022-10-24T00:00:00.000Z,https://scrapingsandbox.com/product/4,2026-11-06T15:30:13.497748
4,5,Smart Carving Tools,SparkleBox,Art,97.13,NaN,High-quality smart carving tools from SparkleB...,SKU-ART-0005,True,3.5,353,"drawing, painting",2022-08-07T00:00:00.000Z,https://scrapingsandbox.com/product/5,2026-11-06T15:30:13.852708
5,6,Professional Dog Food,StyleCraft,Pet Supplies,145.40,NaN,High-quality professional dog food from StyleC...,SKU-PET-0006,True,4.6,156,"cats, grooming, food",2022-08-30T00:00:00.000Z,https://scrapingsandbox.com/product/6,2026-11-06T15:30:14.188239
7,8,Premium Bluetooth Speaker,AutoParts Pro,Electronics,108.99,NaN,High-quality premium bluetooth speaker from Au...,SKU-ELE-0008,True,4.5,426,USB,2023-09-04T00:00:00.000Z,https://scrapingsandbox.com/product/8,2026-11-06T15:30:14.879032
10,11,Professional Brush Kit,AutoParts Pro,Art,55.29,NaN,High-quality professional brush kit from AutoP...,SKU-ART-0011,False,3.6,204,supplies,2023-05-21T00:00:00.000Z,https://scrapingsandbox.com/product/11,2026-11-06T15:30:15.879554
12,13,Premium Vinyl Record,DeskMate,Music,69.22,NaN,High-quality premium vinyl record from DeskMat...,SKU-MUS-0013,True,4.5,442,"instruments, audio, recording",2022-01-19T00:00:00.000Z,https://scrapingsandbox.com/product/13,2026-11-06T15:30:16.527978
15,16,Eco-Friendly Protein Bar,UrbanThread,Food & Beverage,76.27,NaN,High-quality eco-friendly protein bar from Urb...,SKU-FOO-0016,True,3.9,47,organic,2023-05-28T00:00:00.000Z,https://scrapingsandbox.com/product/16,2026-11-06T15:30:17.702747
16,17,Lightweight Paper Clips,UrbanThread,Office Supplies,156.17,NaN,High-quality lightweight paper clips from Urba...,SKU-OFF-0017,True,4.3,80,"stationery, desk, organization",2023-08-21T00:00:00.000Z,https://scrapingsandbox.com/product/17,2026-11-06T15:30:18.034674
18,19,Premium Business Book,PageTurner,Books,175.14,NaN,High-quality premium business book from PageTu...,SKU-BOO-0019,True,4.9,108,paperback,2023-09-21T00:00:00.000Z,https://scrapingsandbox.com/product/19,2026-11-06T15:30:18.695513
19,20,Handcrafted Locket,DeskMate,Jewelry,106.84,NaN,High-quality handcrafted locket from DeskMate....,SKU-JEW-0020,True,4.2,239,"luxury, handmade",2023-06-07T00:00:00.000Z,https://scrapingsandbox.com/product/20,2026-11-06T15:30:19.018105


### Step 7: Check Original Price

In [26]:
print("Products with Original Price:")
print(products_df["Original_Price"].notna().sum())

print("\nProducts without Original Price:")
print(products_df["Original_Price"].isna().sum())

print("\nDiscounted Products:")
print(
    products_df[
        products_df["Original_Price"] > products_df["Price"]
    ].shape[0]
)

Products with Original Price:
189

Products without Original Price:
311

Discounted Products:
189


### Create a separate copy for analysis

In [27]:
analysis_df = products_df.copy()

analysis_df["Has_Original_Price"] = (
    analysis_df["Original_Price"].notna()
)

analysis_df["Discount_Amount"] = (
    analysis_df["Original_Price"] - analysis_df["Price"]
)

analysis_df.loc[
    analysis_df["Original_Price"].isna(),
    "Discount_Amount"
] = None

display(
    analysis_df[
        [
            "Product_Name",
            "Price",
            "Original_Price",
            "Has_Original_Price",
            "Discount_Amount"
        ]
    ].head(10)
)

,Product_Name,Price,Original_Price,Has_Original_Price,Discount_Amount
0,Lightweight Probiotics,155.62,206.69,True,51.07
1,Wireless LED Desk Lamp,145.40,210.01,True,64.61
2,Deluxe Tea Collection,185.01,313.98,True,128.97
3,Essential Tire Gauge,70.86,NaN,False,NaN
4,Smart Carving Tools,97.13,NaN,False,NaN
5,Professional Dog Food,145.40,NaN,False,NaN
6,Smart Car Wash Kit,139.94,173.26,True,33.32
7,Premium Bluetooth Speaker,108.99,NaN,False,NaN
8,Vintage Tablet Stand,44.15,69.14,True,24.99
9,Portable Marker Pack,193.91,258.48,True,64.57


In [28]:
print("Total Products:", len(analysis_df))

print(
    "Products with Original Price:",
    analysis_df["Has_Original_Price"].sum()
)

print(
    "Products without Original Price:",
    (~analysis_df["Has_Original_Price"]).sum()
)

print(
    "Products with Discount:",
    (analysis_df["Discount_Amount"] > 0).sum()
)

print(
    "Invalid Discount Values:",
    (analysis_df["Discount_Amount"] < 0).sum()
)

Total Products: 500
Products with Original Price: 189
Products without Original Price: 311
Products with Discount: 189
Invalid Discount Values: 0


### Data Cleaning & Standardization

#### Step 1 — Create Clean Dataset

In [29]:
clean_df = analysis_df.copy()

print("Original Analysis Dataset:", analysis_df.shape)
print("New Cleaning Dataset:", clean_df.shape)

Original Analysis Dataset: (500, 17)
New Cleaning Dataset: (500, 17)


#### Step 2 — Check Data Types

In [30]:
clean_df.info()

print("\nData Types:")
display(clean_df.dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Product_ID          500 non-null    int64  
 1   Product_Name        500 non-null    str    
 2   Vendor              500 non-null    str    
 3   Category            500 non-null    str    
 4   Price               500 non-null    float64
 5   Original_Price      189 non-null    float64
 6   Description         500 non-null    str    
 7   SKU                 500 non-null    str    
 8   Availability        500 non-null    bool   
 9   Rating              500 non-null    float64
 10  Reviews_Count       500 non-null    int64  
 11  Tags                500 non-null    str    
 12  Created_Date        500 non-null    str    
 13  Product_URL         500 non-null    str    
 14  Scraped_Date        500 non-null    str    
 15  Has_Original_Price  500 non-null    bool   
 16  Discount_Amount    

Product_ID              int64
Product_Name              str
Vendor                    str
Category                  str
Price                 float64
Original_Price        float64
Description               str
SKU                       str
Availability             bool
Rating                float64
Reviews_Count           int64
Tags                      str
Created_Date              str
Product_URL               str
Scraped_Date              str
Has_Original_Price       bool
Discount_Amount       float64
dtype: object

#### Step 3 — Convert Data Types

In [31]:
# Convert numeric columns
clean_df["Product_ID"] = clean_df["Product_ID"].astype("int64")
clean_df["Price"] = pd.to_numeric(clean_df["Price"])
clean_df["Original_Price"] = pd.to_numeric(
    clean_df["Original_Price"]
)
clean_df["Rating"] = pd.to_numeric(clean_df["Rating"])
clean_df["Reviews_Count"] = clean_df["Reviews_Count"].astype("int64")
clean_df["Discount_Amount"] = pd.to_numeric(
    clean_df["Discount_Amount"]
)

# Convert date columns
clean_df["Created_Date"] = pd.to_datetime(
    clean_df["Created_Date"],
    errors="coerce",
    utc=True
)

clean_df["Scraped_Date"] = pd.to_datetime(
    clean_df["Scraped_Date"],
    errors="coerce",
    utc=True
)

# Convert availability to Boolean
clean_df["Availability"] = clean_df["Availability"].astype("bool")

# Convert text columns to string
text_columns = [
    "Product_Name",
    "Vendor",
    "Category",
    "Description",
    "SKU",
    "Tags",
    "Product_URL"
]

for column in text_columns:
    clean_df[column] = clean_df[column].astype("string")

print("Data Types After Conversion:")
display(clean_df.dtypes)

Data Types After Conversion:


Product_ID                          int64
Product_Name                       string
Vendor                             string
Category                           string
Price                             float64
Original_Price                    float64
Description                        string
SKU                                string
Availability                         bool
Rating                            float64
Reviews_Count                       int64
Tags                               string
Created_Date          datetime64[us, UTC]
Product_URL                        string
Scraped_Date          datetime64[us, UTC]
Has_Original_Price                   bool
Discount_Amount                   float64
dtype: object

#### Step 4 — Verify the Conversion

In [32]:
print("Created_Date Type:", clean_df["Created_Date"].dtype)
print("Scraped_Date Type:", clean_df["Scraped_Date"].dtype)

print("\nMissing Values After Conversion:")
display(clean_df.isnull().sum())

Created_Date Type: datetime64[us, UTC]
Scraped_Date Type: datetime64[us, UTC]

Missing Values After Conversion:


Product_ID              0
Product_Name            0
Vendor                  0
Category                0
Price                   0
Original_Price        311
Description             0
SKU                     0
Availability            0
Rating                  0
Reviews_Count           0
Tags                    0
Created_Date            0
Product_URL             0
Scraped_Date            0
Has_Original_Price      0
Discount_Amount       311
dtype: int64

#### Step 5 — Remove Extra Spaces

In [33]:
# Text columns to standardize
text_columns = [
    "Product_Name",
    "Vendor",
    "Category",
    "Description",
    "SKU",
    "Tags"
]

# Remove leading, trailing, and repeated spaces
for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Extra spaces removed successfully.")

Extra spaces removed successfully.


#### Step 6 — Standardize Text Formatting

In [34]:
clean_df["Product_Name"] = clean_df["Product_Name"].str.title()
clean_df["Vendor"] = clean_df["Vendor"].str.title()
clean_df["Category"] = clean_df["Category"].str.title()

print("Text formatting standardized successfully.")

display(
    clean_df[
        ["Product_Name", "Vendor", "Category", "Tags"]
    ].head(10)
)

Text formatting standardized successfully.


,Product_Name,Vendor,Category,Tags
0,Lightweight Probiotics,Soundwave,Health,"natural, fitness"
1,Wireless Led Desk Lamp,Soundwave,Electronics,"tech, digital, USB"
2,Deluxe Tea Collection,Stylecraft,Food & Beverage,organic
3,Essential Tire Gauge,Techvault,Automotive,"car, accessories, travel"
4,Smart Carving Tools,Sparklebox,Art,"drawing, painting"
5,Professional Dog Food,Stylecraft,Pet Supplies,"cats, grooming, food"
6,Smart Car Wash Kit,Urbanthread,Automotive,car
7,Premium Bluetooth Speaker,Autoparts Pro,Electronics,USB
8,Vintage Tablet Stand,Glowup,Electronics,tech
9,Portable Marker Pack,Vitallife,Art,"supplies, creative, crafts"


#### Step 7 — Check Unique Values

In [35]:
print("Number of Unique Vendors:", clean_df["Vendor"].nunique())
print("Number of Unique Categories:", clean_df["Category"].nunique())

print("\nVendors:")
print(sorted(clean_df["Vendor"].unique()))

print("\nCategories:")
print(sorted(clean_df["Category"].unique()))

Number of Unique Vendors: 20
Number of Unique Categories: 15

Vendors:
['Artisanhub', 'Autoparts Pro', 'Cozycorner', 'Deskmate', 'Fitpro', 'Glowup', 'Homenest', 'Mindbooks', 'Novatech', 'Pageturner', 'Pawpal', 'Playzone', 'Progear', 'Soundwave', 'Sparklebox', 'Stylecraft', 'Tastebud', 'Techvault', 'Urbanthread', 'Vitallife']

Categories:
['Art', 'Automotive', 'Beauty', 'Books', 'Clothing', 'Electronics', 'Food & Beverage', 'Health', 'Home & Garden', 'Jewelry', 'Music', 'Office Supplies', 'Pet Supplies', 'Sports', 'Toys']


#### Step 8 — Check Text Quality

In [36]:
for column in ["Product_Name", "Vendor", "Category", "SKU", "Tags"]:
    extra_spaces = clean_df[column].str.contains(
        r"^\s|\s$|\s{2,}",
        regex=True,
        na=False
    ).sum()

    print(f"{column}: {extra_spaces} rows with extra spaces")

Product_Name: 0 rows with extra spaces
Vendor: 0 rows with extra spaces
Category: 0 rows with extra spaces
SKU: 0 rows with extra spaces
Tags: 0 rows with extra spaces


### Numeric Validation

#### Step 9 — Validate Product Prices

In [37]:
print("Prices <= 0:", (clean_df["Price"] <= 0).sum())

print(
    "Original Prices <= 0:",
    (clean_df["Original_Price"].dropna() <= 0).sum()
)

print(
    "Original Price lower than Current Price:",
    (
        clean_df["Original_Price"].notna()
        & (clean_df["Original_Price"] < clean_df["Price"])
    ).sum()
)

print(
    "Negative Discount Amounts:",
    (clean_df["Discount_Amount"] < 0).sum()
)

Prices <= 0: 0
Original Prices <= 0: 0
Original Price lower than Current Price: 0
Negative Discount Amounts: 0


#### Step 10 — Validate Ratings and Reviews

In [38]:
print(
    "Ratings below 0:",
    (clean_df["Rating"] < 0).sum()
)

print(
    "Ratings above 5:",
    (clean_df["Rating"] > 5).sum()
)

print(
    "Negative Reviews Count:",
    (clean_df["Reviews_Count"] < 0).sum()
)

Ratings below 0: 0
Ratings above 5: 0
Negative Reviews Count: 0


#### Step 11 — Statistical Summary

In [39]:
numeric_columns = [
    "Price",
    "Original_Price",
    "Rating",
    "Reviews_Count",
    "Discount_Amount"
]

display(clean_df[numeric_columns].describe().round(2))

,Price,Original_Price,Rating,Reviews_Count,Discount_Amount
count,500.00,189.00,500.00,500.00,189.00
mean,104.10,161.36,4.02,234.27,50.60
std,58.27,88.92,0.59,141.34,32.07
min,5.12,7.76,3.00,0.00,1.54
25%,55.30,80.02,3.50,109.75,23.58
50%,105.23,175.37,4.05,228.00,49.81
75%,152.62,237.51,4.60,337.25,70.03
max,204.86,330.47,5.00,499.00,133.80


#### Step 12 — Check Discount Consistency

In [40]:
discount_check = (
    clean_df["Original_Price"].notna()
    & (
        clean_df["Discount_Amount"]
        != clean_df["Original_Price"] - clean_df["Price"]
    )
)

print("Inconsistent Discount Records:", discount_check.sum())

Inconsistent Discount Records: 0


### Final Data Quality Check

#### Step 13 — Final Dataset Validation

In [41]:
print("Dataset Shape:", clean_df.shape)

print("\nDuplicate Rows:",
      clean_df.duplicated().sum())

print("Duplicate Product IDs:",
      clean_df["Product_ID"].duplicated().sum())

print("Duplicate SKUs:",
      clean_df["SKU"].duplicated().sum())

print("Missing Product Names:",
      clean_df["Product_Name"].isna().sum())

print("Missing Product IDs:",
      clean_df["Product_ID"].isna().sum())

print("Invalid URLs:",
      (~clean_df["Product_URL"].str.startswith("https://")).sum())

print("Invalid Ratings:",
      ((clean_df["Rating"] < 0) | (clean_df["Rating"] > 5)).sum())

print("Negative Prices:",
      (clean_df["Price"] < 0).sum())

Dataset Shape: (500, 17)

Duplicate Rows: 0
Duplicate Product IDs: 0
Duplicate SKUs: 0
Missing Product Names: 0
Missing Product IDs: 0
Invalid URLs: 0
Invalid Ratings: 0
Negative Prices: 0


#### Step 14 — Verify Product and Variant Relationship

In [42]:
product_ids = set(clean_df["Product_ID"])
variant_product_ids = set(variants_df["Product_ID"])

orphan_variants = variants_df[
    ~variants_df["Product_ID"].isin(product_ids)
]

print("Total Products:", len(product_ids))
print("Total Variants:", len(variants_df))
print("Orphan Variants:", len(orphan_variants))
print(
    "Products without Variants:",
    len(product_ids - variant_product_ids)
)

Total Products: 500
Total Variants: 1685
Orphan Variants: 0
Products without Variants: 0


### 3. Export Final Clean Dataset

In [43]:
clean_df.to_csv(
    "clean_ecommerce_products.csv",
    index=False,
    encoding="utf-8-sig"
)

variants_df.to_csv(
    "ecommerce_variants.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Clean datasets exported successfully!")

Clean datasets exported successfully!
